In [1]:
from ultralytics import YOLO
from pathlib import Path
import torch
import pandas as pd

PROJ = Path(r"D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理").resolve()
YOLO_ROOT = PROJ / "yolo_dataset_process"
IMG_ALL   = YOLO_ROOT / "yolo_dataset" / "images"   # YOLO 全部影像
GT_ALL    = PROJ / "gt_all.csv"                    # 上一個 notebook 做好的

best = YOLO_ROOT/"runs"/"detect"/"train_nb"/"weights"/"best.pt"  # 用你最好那次
assert best.exists(), best

model = YOLO(str(best))
device = "0" if torch.cuda.is_available() else "cpu"
print("device =", device)

df_gt = pd.read_csv(GT_ALL)
# 這邊只留 filename/side/stage 當 metadata，用來接在 pred 後面
meta = df_gt[["filename", "side", "stage"]].drop_duplicates()
meta.head()


device = 0


,filename,side,stage
0,S1_L1.jpg,L,1
2,S1_L10.jpg,L,1
4,S1_L11.jpg,L,1
6,S1_L12.jpg,L,1
7,S1_L13.jpg,L,1


In [2]:
rows = []

# conf 門檻：太低會很多奇怪框，建議先 0.25~0.4
CONF_TH = 0.25

for img in sorted(IMG_ALL.glob("*.*")):
    if img.suffix.lower() not in [".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff"]:
        continue

    res = model.predict(
        source=str(img),
        conf=CONF_TH,
        device=device,
        imgsz=640,
        verbose=False
    )[0]

    if res.boxes is None or len(res.boxes) == 0:
        # 這張圖沒偵測到，就先記一筆空的，之後可分析
        rows.append({
            "filename": img.name,
            "x1": None, "y1": None, "x2": None, "y2": None,
            "score": None
        })
        continue

    # 取出所有 YOLO 預測框
    boxes  = res.boxes.xyxy.cpu().numpy()
    scores = res.boxes.conf.cpu().numpy()

    # 從 meta 找這張圖的 side（L / R）
    row_meta = meta[meta["filename"] == img.name]
    side = row_meta.iloc[0]["side"] if len(row_meta) > 0 else None  # 可能是 'L' 或 'R'

    # 判斷要選哪一顆框
    if side == "L" and boxes.shape[0] > 0:
    # 病人左髖在畫面右邊 → 選 x2 最大（最靠右）
        k = boxes[:, 2].argmax()
    elif side == "R" and boxes.shape[0] > 0:
        # 病人右髖在畫面左邊 → 選 x1 最小（最靠左）
        k = boxes[:, 0].argmin()
    else:
        # side 不明或沒框時的保險
        k = scores.argmax()


    x1, y1, x2, y2 = boxes[k]
    sc = scores[k]

    rows.append({
        "filename": img.name,
        "x1": float(x1), "y1": float(y1),
        "x2": float(x2), "y2": float(y2),
        "score": float(sc),
    })

df_pred_best = pd.DataFrame(rows)
print("pred rows:", len(df_pred_best))
df_pred_best.head()


pred rows: 287


,filename,x1,y1,x2,y2,score
0,S1_L1.jpg,1070.583496,209.588837,1340.963379,555.759399,0.924170
1,S1_L10.jpg,475.121246,244.086227,749.309509,597.667114,0.862840
2,S1_L11.jpg,1017.394470,180.653488,1349.964600,584.668274,0.897389
3,S1_L12.jpg,1073.150146,172.534256,1378.838135,596.896729,0.914369
4,S1_L13.jpg,1096.489014,185.280121,1382.047363,543.120361,0.900219


In [3]:
# 跟 meta (filename, side, stage) merge
df_roi = df_pred_best.merge(meta, on="filename", how="left")

ROI_CSV = PROJ / "roi_all.csv"
df_roi.to_csv(ROI_CSV, index=False, encoding="utf-8-sig")
print("✅ 輸出 roi_all.csv >", ROI_CSV)
df_roi.head()


✅ 輸出 roi_all.csv > D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\roi_all.csv


,filename,x1,y1,x2,y2,score,side,stage
0,S1_L1.jpg,1070.583496,209.588837,1340.963379,555.759399,0.924170,L,1
1,S1_L10.jpg,475.121246,244.086227,749.309509,597.667114,0.862840,L,1
2,S1_L11.jpg,1017.394470,180.653488,1349.964600,584.668274,0.897389,L,1
3,S1_L12.jpg,1073.150146,172.534256,1378.838135,596.896729,0.914369,L,1
4,S1_L13.jpg,1096.489014,185.280121,1382.047363,543.120361,0.900219,L,1


In [4]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# 如果剛上面已經有 df_roi / ROI_CSV，就不一定要重讀
df_roi = pd.read_csv(ROI_CSV)

IMG_ALL = YOLO_ROOT / "yolo_dataset" / "images"   # 全部影像
OUT_DIR = PROJ / "vis_all_roi_side"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("輸出資料夾：", OUT_DIR)

n_ok, n_miss_det, n_miss_meta = 0, 0, 0

for img_path in sorted(IMG_ALL.glob("*.*")):
    if img_path.suffix.lower() not in [".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff"]:
        continue

    fn = img_path.name
    img = cv2.imread(str(img_path))
    if img is None:
        print("讀圖失敗:", img_path)
        continue

    # 找這張圖在 roi_all 裡的資料
    row = df_roi[df_roi["filename"] == fn]
    if len(row) == 0:
        # 沒有 meta / roi 資訊
        n_miss_meta += 1
        out = img.copy()
        cv2.putText(out, "NO META", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA)
        cv2.imwrite(str(OUT_DIR / fn), out)
        continue

    row = row.iloc[0]

    # 如果這張圖沒有偵測到（x1 是 NaN）
    if pd.isna(row["x1"]) or pd.isna(row["y1"]) or pd.isna(row["x2"]) or pd.isna(row["y2"]):
        n_miss_det += 1
        out = img.copy()
        # 標上 side / stage + NO DET
        txt = f"side={row.get('side', '?')} stage={row.get('stage', '?')}  NO DET"
        cv2.putText(out, txt, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)
        cv2.imwrite(str(OUT_DIR / fn), out)
        continue

    # 正常有偵測到，畫框
    x1, y1, x2, y2 = map(int, [row["x1"], row["y1"], row["x2"], row["y2"]])
    score = row.get("score", None)
    side  = row.get("side", "?")
    stage = row.get("stage", "?")

    out = img.copy()

    # 畫框（綠色）
    cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # 標註文字：side / stage / score
    label = f"side={side} stage={stage}"
    if score is not None and not np.isnan(score):
        label += f" score={score:.2f}"

    cv2.putText(out, label, (x1, max(0, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

    cv2.imwrite(str(OUT_DIR / fn), out)
    n_ok += 1

print("✅ 完成可視化")
print("  正常有框的張數   :", n_ok)
print("  無偵測結果 (NO DET):", n_miss_det)
print("  找不到 meta/roi 的 :", n_miss_meta)
print("  請到資料夾查看：", OUT_DIR)


輸出資料夾： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\vis_all_roi_side
✅ 完成可視化
  正常有框的張數   : 287
  無偵測結果 (NO DET): 0
  找不到 meta/roi 的 : 0
  請到資料夾查看： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\vis_all_roi_side
